In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_28.pickle

In [ ]:
%%PandasProfile
### cell 28 ###

def grab_subset_of_data_40(original_df, question_of_interest):
    # Vectorized column filtering and renaming without explicit copy
    return (
        original_df
        .filter(like=question_of_interest)
        .rename(columns=lambda col: col.rsplit('-', 1)[-1].lstrip())
        .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest, include_2017=False):
    # Define sources in chronological order
    sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10)
    ]
    if include_2017:
        sources.insert(0, ('2017', responses_df_2017))

    # Subset, annotate year, and collect
    dfs = []
    for year, df_src in sources:
        df_sub = grab_subset_of_data_40(df_src, question_of_interest)
        if not df_sub.empty:
            dfs.append(df_sub.assign(year=year))

    # Concatenate and count non-nulls per year
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined_counts = df_combined.groupby('year', sort=True).count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_40(df, df_counts):
    # Compute total respondents per year once
    years = ['2018', '2019', '2020', '2021', '2022']
    totals = df['year'].value_counts().reindex(years).fillna(0).astype(int)

    # Convert counts to percentages in a vectorized manner
    pct = (
        df_counts
        .set_index('year')
        .astype(int)
        .div(totals, axis=0)
        .mul(100)
        .reindex(years)
        .reset_index()
    )
    return pct

# Re-apply column renaming to unify question texts
q_orig = 'Which of the following hosted notebook products do you use on a regular basis?'
q_new = 'Do you use any of the following hosted notebook products?'
responses_df_2021.columns = responses_df_2021.columns.str.replace(q_orig, q_new, regex=False)
responses_df_2020.columns = responses_df_2020.columns.str.replace(q_orig, q_new, regex=False)

# Rename specific answer texts in 2019 and 2018 dataframes
replacements_2019 = {
    'Google Colab ': 'Colab Notebooks',
    'Kaggle Notebooks (Kernels) ': 'Kaggle Notebooks'
}
for old, new in replacements_2019.items():
    responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(old, new, regex=False)
    responses_df_2019_cell10.replace(old, new, regex=False, inplace=True)

replacements_2018 = {
    'Google Colab': 'Colab Notebooks',
    'Kaggle Kernels': 'Kaggle Notebooks'
}
for old, new in replacements_2018.items():
    responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(old, new, regex=False)
    responses_df_2018_cell10.replace(old, new, regex=False, inplace=True)

# Combine, convert, reshape
question_of_interest_cell40 = 'Do you use any of the following hosted notebook products?'
notebooks_df_combined, notebooks_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
notebooks_df_combined_percentages = \
    convert_df_of_counts_to_percentages_40(
        notebooks_df_combined,
        notebooks_df_combined_counts
    )

# Select and melt
df_cell40 = (
    notebooks_df_combined_percentages
    .loc[:, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']]
    .melt(id_vars='year', value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks'])
    .rename(columns={'variable': ''})
)

df_cell40.info()